# 00 — Preflight: verify paths, GPU, packages

**Run this first.** These checks do not edit old project files, train, or quantize models. Read `README.md` for the overall workflow.

## Step 1 — Locate the new standalone folder

No path edits are normally needed because files are resolved relative to this folder.

In [1]:
from pathlib import Path
import subprocess, sys, json

def find_analysis_folder():
    current = Path.cwd().resolve()
    for parent in [current, *current.parents]:
        for relative in [Path('notebooks/runtime_analysis'), Path('runtime_analysis'), Path('.')]:
            candidate = (parent / relative).resolve()
            if (candidate / 'config.json').is_file() and (candidate / 'worker.py').is_file():
                return candidate
    raise FileNotFoundError('Cannot locate notebooks/runtime_analysis/config.json. Open this notebook inside the copied repository.')

ANALYSIS_DIR = find_analysis_folder()
REPO_ROOT = ANALYSIS_DIR.parent.parent
CONFIG = json.loads((ANALYSIS_DIR/'config.json').read_text())
print('Folder:', ANALYSIS_DIR)
print('Repository:', REPO_ROOT)

Folder: /home/mohnishr/workspace/DLBA/Quantization-Spurious-Heuristics-in-Transformers/notebooks/runtime_analysis
Repository: /home/mohnishr/workspace/DLBA/Quantization-Spurious-Heuristics-in-Transformers


## 🔴 Step 2 — Configuration to check

If model paths differ, edit **`config.json` only**, not the code. Do not alter model precision or HANS evaluation settings when comparing variants.

In [2]:
from pprint import pprint
pprint(CONFIG)
for seed in CONFIG['seeds']:
    checkpoint = REPO_ROOT / CONFIG['model_dir_template'].format(seed=seed)
    preds = REPO_ROOT / CONFIG['hans_predictions_template'].format(seed=seed)
    print(f"Seed {seed}: FP32 checkpoint {'FOUND' if checkpoint.is_dir() else 'MISSING'} → {checkpoint}")
    print(f"Seed {seed}: cached HANS FP32 CSV {'FOUND' if preds.is_file() else 'MISSING'} → {preds}")

{'batch_sizes': [32],
 'benchmark_repeats': 3,
 'compile_dynamic': True,
 'compile_mode': 'max-autotune-no-cudagraphs',
 'device': 'cuda',
 'hans_predictions_template': 'results/hans/fp32_seed{seed}_predictions.csv',
 'hans_tsv_url': 'https://raw.githubusercontent.com/tommccoy1/hans/master/heuristics_evaluation_set.txt',
 'latency_batches_per_size': 100,
 'max_examples': None,
 'max_length': 128,
 'model_dir_template': 'models/fp32/seed{seed}/final',
 'prediction_probability_tolerance': 0.001,
 'profiling_batches': 8,
 'reference_int8_predictions_template': 'results/hans/int8_seed{seed}_predictions.csv',
 'results_dir': 'results/runtime_analysis',
 'seeds': [42, 123, 456],
 'throughput_batch_size': 32,
 'warmup_batches': 20}
Seed 42: FP32 checkpoint FOUND → /home/mohnishr/workspace/DLBA/Quantization-Spurious-Heuristics-in-Transformers/models/fp32/seed42/final
Seed 42: cached HANS FP32 CSV FOUND → /home/mohnishr/workspace/DLBA/Quantization-Spurious-Heuristics-in-Transformers/results/han

## Step 3 — Validate your Python environment and GPU

Use the *same notebook kernel* in which your original TorchAO quantization successfully ran.

In [3]:
import torch, torchao, transformers, pandas, psutil, numpy
print('PyTorch:', torch.__version__)
print('TorchAO:', torchao.__version__)
print('Transformers:', transformers.__version__)
print('CUDA available:', torch.cuda.is_available())
if CONFIG['device'] == 'cuda':
    assert torch.cuda.is_available(), 'No CUDA available in selected kernel. Fix kernel/CUDA before proceeding.'
    print('GPU:', torch.cuda.get_device_name(0))
    print('CUDA runtime:', torch.version.cuda)
from torchao.quantization import quantize_, Int8DynamicActivationInt8WeightConfig
print('✓ TorchAO configuration imported')

W1025 22:45:32.765000 44374 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


PyTorch: 2.14.0+cu130
TorchAO: 0.18.0
Transformers: 5.17.0
CUDA available: True
GPU: NVIDIA GeForce RTX 5080 Laptop GPU
CUDA runtime: 13.0
✓ TorchAO configuration imported


## Step 4 — Confirm HANS input and row identity

Prefer your saved `results/hans/fp32_seed42_predictions.csv`; if absent, falls back to the original HANS raw TSV URL. **Never** calls `load_dataset("hans")`.

In [4]:
sys.path.insert(0, str(ANALYSIS_DIR))
from runtime_common import input_examples
examples, source = input_examples(CONFIG, 42)
print('Source:', source)
print('Examples:', len(examples))
print('Unique pairID:', examples['pairID'].nunique())
display(examples[['pairID','premise','hypothesis']].head(3))

Source: /home/mohnishr/workspace/DLBA/Quantization-Spurious-Heuristics-in-Transformers/results/hans/fp32_seed42_predictions.csv
Examples: 30000
Unique pairID: 30000


,pairID,premise,hypothesis
0,ex0,The president advised the doctor .,The doctor advised the president .
1,ex1,The student saw the managers .,The managers saw the student .
2,ex2,The presidents encouraged the banker .,The banker encouraged the presidents .


## Next

Open **`01_isolated_eager.ipynb`**. Start with only seed 42. The new scripts put all NEW results under `results/runtime_analysis/`.